In [1]:
import pandas as pd
import random
import numpy as np

In [ ]:
# 計画した列のリスト
columns = ['order_id','order_datetime', 'from_where', 'to_where', 'order_category', 'order_emergency', 'order_timestamp', 'order_value', 'order_status']

# 空のデータフレームを作成
orders_df = pd.DataFrame(columns=columns)

print(orders_df)

Empty DataFrame
Columns: [order_id, order_daytime, from_where, to_where, order_category, order_emergency, order_timestamp, order_value, order_status]
Index: []


In [2]:
import pandas as pd
import random
from datetime import datetime, timedelta
import uuid

def generate_realistic_orders(hospitals_df, supermarkets_df, residences_df, num_orders, start_date=None, time_period='day'):
    """病院、スーパーマーケット、住宅のデータフレームから現実的なオーダーを生成する関数。"""
    hospitals_df['category'] = 'Hospital'
    supermarkets_df['category'] = 'Supermarket'
    residences_df['category'] = 'Residence'
    
    group_a = pd.concat([hospitals_df, supermarkets_df, residences_df], ignore_index=True)
    group_b = pd.concat([hospitals_df, supermarkets_df, residences_df], ignore_index=True)
    
    if start_date is None:
        start_date = datetime.now()
    if time_period == 'day':
        end_date = start_date + timedelta(days=1)
    else:
        raise ValueError("対応していない期間です。現在は'day'のみ対応しています。")

    columns = ['order_id', 'order_datetime', 'from_where', 'to_where', 'order_category', 'order_emergency', 'order_timestamp', 'order_value', 'order_status']
    orders = []
    
    for _ in range(num_orders):
        order_id = str(uuid.uuid4())
        random_delta = random.random() * (end_date - start_date).total_seconds()
        order_datetime = start_date + timedelta(seconds=random_delta)
        
        while True:
            from_location = group_a.sample(1).iloc[0]
            to_location = group_b.sample(1).iloc[0]

            if (from_location['category'] == 'Hospital' and to_location['category'] == 'Supermarket') or \
               (from_location['category'] == 'Supermarket' and to_location['category'] == 'Hospital'):
                continue
            if from_location['name'] == to_location['name']:
                continue
            break
            
        if from_location['category'] == 'Supermarket':
            order_category = 'Pickup'
        elif from_location['category'] == 'Hospital':
            order_category = 'Delivery'
        else:
            order_category = 'Transport'
            
        order_emergency = random.choice(['low', 'medium', 'high'])
        order_value = round(random.uniform(500, 5000), 2)
        order_status = random.choice(['pending', 'in_progress', 'completed', 'cancelled'])

        orders.append({
            'order_id': order_id,
            'order_datetime': order_datetime,
            'from_where': from_location['name'],
            'to_where': to_location['name'],
            'order_category': order_category,
            'order_emergency': order_emergency,
            'order_timestamp': order_datetime,
            'order_value': order_value,
            'order_status': order_status
        })

    orders_df = pd.DataFrame(orders, columns=columns)
    return orders_df

if __name__ == '__main__':
    try:
        hospitals = pd.read_csv('hospitals.csv')
        supermarkets = pd.read_csv('supermarkets.csv')
        residences = pd.read_csv('residences.csv')
        
        # 100件のオーダーを生成し、CSVとして保存
        orders_df = generate_realistic_orders(hospitals, supermarkets, residences, num_orders=100)
        orders_df.to_csv('orders.csv', index=False)
        print("オーダーデータがCSVファイルとしてorders.csvに保存されました。")

    except FileNotFoundError:
        print("CSVファイルが見つかりません。先に generate_map.py を実行してください。")

オーダーデータがCSVファイルとしてorders.csvに保存されました。
